> **SUPERSEDED** — This is an earlier version of the satellite pull notebook. The current notebook (which produced the actual dataset used for model training) is **[AirSentinel_Satellite_Pull_OPTIMIZED.ipynb](AirSentinel_Satellite_Pull_OPTIMIZED.ipynb)** — it adds existence-check-before-pull logic and produced 700 real S2 images. Open that one, not this one.

# AirSentinel — Satellite Data Pull (Final, pre-Prithvi)

**Person 2's working notebook.** This covers everything up to — but not including — fine-tuning Prithvi. By the end of this notebook you will have:

1. A regular satellite "photo" of each of Delhi's 13 hotspot zones, **on every date you choose** (Sentinel-2)
2. A pollution-relevant "heatmap" of each zone on each date, showing nitrogen dioxide levels (Sentinel-5P) — this is the layer that's actually relevant to pollution, not the photo
3. Everything saved neatly in one Google Drive folder
4. A starter table ready to receive real pollution labels from your teammate's government monitoring-station data, so Phase 2 (labeling + fine-tuning Prithvi) has something solid to build on instead of guesswork

Run every cell top to bottom, in order, the first time. Each section explains in plain language what it does and why, before the code that does it.

Before you start: you'll need your Earth Engine **project ID** (not an API key — see Step 2) ready to paste in below.

## Step 1 — Install the libraries

**What this does:** installs the tools this notebook needs — the Earth Engine connector, a mapping helper (`geemap`), and a library for reading the satellite image files (`rasterio`). Run once per session, takes under a minute.

In [1]:
!pip install earthengine-api geemap rasterio -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 13.6 MB/s eta 0:00:0000:0100:01


## Step 2 — Connect to Earth Engine

**What this does:** logs this notebook into your Earth Engine account and points it at your specific project.

Running the cell below pops up a Google sign-in window — sign in with the account you registered Earth Engine with, then click Allow.

**Important:** `PROJECT_ID` is your Earth Engine **project ID** — a short name like `airpollution-501715` — not an API key (those look like a long random string starting with `AIza...` and are a different thing entirely). Find your project ID at code.earthengine.google.com, top-left corner. Double-check there's no extra space inside the quotes when you paste it in — that alone will cause a connection error.

In [3]:
import ee
import geemap

ee.Authenticate()

PROJECT_ID = 'airpollution-501715'  # <-- replace with your actual Earth Engine project ID, no extra spaces
ee.Initialize(project=PROJECT_ID)

print('Connected to Earth Engine, project:', PROJECT_ID)

Connected to Earth Engine, project: airpollution-501715


## Step 3 — Test pull: one point, one image

**What this does:** proves the whole connection works before you build anything on top of it. Pulls the least-cloudy Sentinel-2 (regular satellite photo) image of central Delhi and displays it right here.

**If you see a real satellite picture of Delhi below — you're good, move on.** If you see a striped black-and-yellow "blocked" tile instead, that's OpenStreetMap's free background map blocking Colab (a known, unrelated issue) — the fix is already built into the code below (`basemap='HYBRID'`).

In [4]:
test_point = ee.Geometry.Point([77.2090, 28.6139])  # India Gate, central Delhi — test location only

test_image = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
              .filterBounds(test_point)
              .filterDate('2026-06-01', '2026-07-01')
              .sort('CLOUDY_PIXEL_PERCENTAGE')
              .first())

Map = geemap.Map(basemap='HYBRID')  # 'HYBRID' avoids OpenStreetMap's tile servers entirely
Map.centerObject(test_point, 11)
Map.addLayer(test_image, {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000}, 'Sentinel-2 test image')
Map

TimeoutException: Requesting secret GOOGLE_MAPS_API_KEY timed out. Secrets can only be fetched when running from the Colab UI.

Common errors here and what they mean:
- `NameError: name 'ee' is not defined` — you skipped Step 2, or your Colab session restarted. Re-run Step 2, then this cell.
- `EEException: Project ... not found` — your `PROJECT_ID` is wrong, has a typo, or has a stray space in the quotes.
- A striped "Referer is required" tile — harmless background-map issue, already fixed by `basemap='HYBRID'` above.

## Step 4 — Define the 13 Delhi hotspot zones

**What this does:** sets up the list of zones everything else in this notebook loops over. These are DPCC's officially declared pollution hotspots.

**Coordinates are now verified, not guessed.** These match the actual official CPCB CAAQMS monitoring station locations for each DPCC station (source: CPCB's national station list / app.cpcbccr.com) -- not an estimated locality centre. The old guessed coordinates were off by anywhere from under 100m up to 6.5km (Rohini was the worst case) -- worth knowing if you already pulled images using the old coordinates, since those were centred on the wrong spot for a few zones.

In [5]:
zones = {
    'Anand Vihar':   [77.315809, 28.647622],
    'Mundka':        [77.076574, 28.684678],
    'Wazirpur':      [77.165453, 28.699793],
    'Jahangirpuri':  [77.170633, 28.732820],
    'RK Puram':      [77.186937, 28.563262],
    'Rohini':        [77.119920, 28.732528],
    'Punjabi Bagh':  [77.131023, 28.674045],
    'Okhla':         [77.271255, 28.530785],
    'Bawana':        [77.051074, 28.776200],
    'Vivek Vihar':   [77.315260, 28.672342],
    'Narela':        [77.101981, 28.822836],
    'Ashok Vihar':   [77.181665, 28.695381],
    'Dwarka':        [77.071901, 28.571027],
}

print(f'{len(zones)} zones defined, using verified CPCB CAAQMS station coordinates.')

13 zones defined, using verified CPCB CAAQMS station coordinates.


## Step 4.5 - Choose which dates to pull (this is what gives you MULTIPLE samples per zone)

**What this does, and why it matters:** everything before this pulled exactly one image per zone, total. That's enough to prove the pipeline works, but nowhere near enough to actually fine-tune a model on later.

This cell fixes that by **auto-generating a list of dates** instead of you typing each one by hand. Set `NUM_DATES` and `DATE_SPACING_DAYS` below, and it builds the full list for you, starting from `BASE_DATE`.

**Default: 10 dates, spaced 5 days apart -> 13 zones x 10 dates = up to 130 images.** 5-day spacing is deliberate, not arbitrary: Sentinel-2 revisits the same spot roughly every 5 days, so spacing any closer than that risks pulling the exact same underlying image twice under two different date labels.

**To change how much data you pull:** just edit `NUM_DATES` (more dates = more samples, but also more Earth Engine usage and longer export/runtime) or `DATE_SPACING_DAYS` (smaller spacing = dates closer together, useful if you want a tighter time window instead of spreading across months).

**`WINDOW_DAYS`** still works the same as before -- for each generated date, it searches a small window around it (default: 3 days before/after) and picks the least-cloudy image in that window, so a single cloudy day doesn't cost you that whole data point.

In [6]:
from datetime import datetime, timedelta

# Edit these three to control how many dates you pull, and how spread out they are
NUM_DATES = 10
DATE_SPACING_DAYS = 5   # 5 days matches Sentinel-2's revisit cycle -- closer spacing risks duplicate images
BASE_DATE = '2026-06-01'

base = datetime.strptime(BASE_DATE, '%Y-%m-%d')
TARGET_DATES = [(base + timedelta(days=i * DATE_SPACING_DAYS)).strftime('%Y-%m-%d') for i in range(NUM_DATES)]

WINDOW_DAYS = 3  # how many days before/after each target date to search for a clean image

def date_window(date_str, window_days=WINDOW_DAYS):
    # Given a target date, returns a (start, end) string range around it for Earth Engine filtering.
    center = datetime.strptime(date_str, '%Y-%m-%d')
    start = (center - timedelta(days=window_days)).strftime('%Y-%m-%d')
    end = (center + timedelta(days=window_days)).strftime('%Y-%m-%d')
    return start, end

print(f'{len(TARGET_DATES)} target dates generated: {TARGET_DATES[0]} to {TARGET_DATES[-1]}')
print(f'Total zone x date combinations: {len(zones) * len(TARGET_DATES)}')

10 target dates generated: 2026-06-01 to 2026-07-16
Total zone x date combinations: 130


## Step 4.6 - Add historical crop-burning-season dates

**What this does, and why it exists:** every date so far has been in 2026, during summer -- which means one whole category your model needs to learn, `crop_burning_smoke`, has zero real examples. That's not a data-volume problem, it's a *season* problem: Delhi's crop-burning season is Oct-Nov, and that hasn't happened yet in 2026. No amount of pulling more summer dates fixes this.

**The fix:** pull real satellite images from Oct-Nov of a *past* year instead -- 2025's burning season already happened, so Earth Engine has real coverage of it. These dates get added to your existing `TARGET_DATES` list, so every cell below (pulling, exporting, sorting, previewing, labeling) picks them up automatically -- nothing else needs to change.

**One thing to check before relying on this:** your teammate's CAAQMS scraper needs to be able to reach *historical* dates (Oct-Nov 2025), not just live/current data, for the heuristic labeler to have real pollution readings to work from for these dates. Confirm that before assuming these images will come with usable labels.

In [7]:
# Real Oct-Nov 2025 dates -- Delhi's actual last crop-burning season, weekly through the peak window
HISTORICAL_DATES = [
    '2025-10-01', '2025-10-08', '2025-10-15', '2025-10-22', '2025-10-29',
    '2025-11-05', '2025-11-12', '2025-11-19', '2025-11-26',
]

before_count = len(TARGET_DATES)
TARGET_DATES.extend(d for d in HISTORICAL_DATES if d not in TARGET_DATES)
added = len(TARGET_DATES) - before_count

print(f'Added {added} historical date(s). TARGET_DATES now has {len(TARGET_DATES)} total.')
print(f'Total zone x date combinations: {len(zones) * len(TARGET_DATES)}')
print('\nNow re-run Steps 5, 7, 8, 8.5, 9, and 10 to pull, export, sort, preview, and label these new dates.')

Added 9 historical date(s). TARGET_DATES now has 19 total.
Total zone x date combinations: 247

Now re-run Steps 5, 7, 8, 8.5, 9, and 10 to pull, export, sort, preview, and label these new dates.


## Step 5 - Pull the "photo" layer (Sentinel-2) for each zone, for each date

**What this does:** for every zone, and for every date in `TARGET_DATES`, grabs the least-cloudy regular satellite photo available within that date's search window. This is what a zone *looks like* -- buildings, roads, green cover. **It does not show pollution** -- think of it as context, not evidence.

Results are stored keyed by `(zone, date)`, so `zone_images_s2[('Anand Vihar', '2026-06-01')]` gets you that specific zone-date's image.

If a zone+date combo prints "no images found," every image in that window was too cloudy to use -- increase `WINDOW_DAYS` in Step 4.5 and re-run.

In [9]:
zone_images_s2 = {}

for name, coords in zones.items():
    point = ee.Geometry.Point(coords)

    for date in TARGET_DATES:
        start, end = date_window(date)
        collection = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
                      .filterBounds(point)
                      .filterDate(start, end)
                      .sort('CLOUDY_PIXEL_PERCENTAGE'))

        count = collection.size().getInfo()
        if count == 0:
            print(f'{name:15s} {date} -- no Sentinel-2 images found in this window, try a wider WINDOW_DAYS')
            continue

        image = collection.first()
        cloud_pct = image.get('CLOUDY_PIXEL_PERCENTAGE').getInfo()
        zone_images_s2[(name, date)] = image
        print(f'{name:15s} {date} -- photo found, {cloud_pct:.1f}% cloudy, {count} candidates in window')

print(f'\nTotal S2 images pulled: {len(zone_images_s2)} out of {len(zones) * len(TARGET_DATES)} possible zone-date combinations')

Anand Vihar     2026-06-01 -- photo found, 26.4% cloudy, 1 candidates in window
Anand Vihar     2026-06-06 -- photo found, 0.0% cloudy, 2 candidates in window
Anand Vihar     2026-06-11 -- photo found, 78.5% cloudy, 1 candidates in window
Anand Vihar     2026-06-16 -- photo found, 0.0% cloudy, 1 candidates in window
Anand Vihar     2026-06-21 -- photo found, 46.9% cloudy, 1 candidates in window
Anand Vihar     2026-06-26 -- photo found, 0.5% cloudy, 2 candidates in window
Anand Vihar     2026-07-01 -- photo found, 98.3% cloudy, 1 candidates in window
Anand Vihar     2026-07-06 -- photo found, 56.9% cloudy, 1 candidates in window
Anand Vihar     2026-07-11 -- photo found, 0.1% cloudy, 1 candidates in window
Anand Vihar     2026-07-16 -- photo found, 55.3% cloudy, 2 candidates in window
Anand Vihar     2025-10-01 -- photo found, 97.6% cloudy, 1 candidates in window
Anand Vihar     2025-10-08 -- photo found, 1.4% cloudy, 2 candidates in window
Anand Vihar     2025-10-15 -- photo found, 0.

## Step 6 - Preview any zone+date photo on an interactive map

**What this does:** lets you look at any one zone, on any one date, right in the notebook, on a zoomable map. Change `zone_to_view` and `date_to_view` to any combination you actually pulled in Step 5 (check the printed output above for which ones succeeded).

In [8]:
zone_to_view = 'Anand Vihar'
date_to_view = TARGET_DATES[0]  # change to any date from TARGET_DATES that printed successfully above

key = (zone_to_view, date_to_view)
if key not in zone_images_s2:
    print(f'No image found for {key} -- pick a zone/date combo that succeeded in Step 5\'s output above.')
else:
    Map2 = geemap.Map(basemap='HYBRID')
    Map2.centerObject(ee.Geometry.Point(zones[zone_to_view]), 13)
    Map2.addLayer(zone_images_s2[key], {'bands': ['B4', 'B3', 'B2'], 'min': 0, 'max': 3000}, f'{zone_to_view} {date_to_view}')
    display(Map2)

NameError: name 'zone_images_s2' is not defined

## Step 7 - Pull the "pollution" layer (Sentinel-5P) for each zone, for each date

**What this does:** same idea as Step 5, but for the actual pollution-relevant layer. For every zone and every date in `TARGET_DATES`, this measures nitrogen dioxide (NO2) levels using that date's search window, averaging whatever daily readings fall inside it into one steadier number.

**Two honest caveats, same as before:**
- Averaging across the window smooths out daily noise, but also means this isn't a single precise moment -- it's a "typical reading around this date."
- Sentinel-5P's resolution is much coarser than Sentinel-2's -- each reading represents several kilometres, not fine detail.

In [ ]:
zone_images_no2 = {}

for name, coords in zones.items():
    point = ee.Geometry.Point(coords)
    region = point.buffer(5000).bounds()  # ~5km around the zone centre, to suit S5P's coarser resolution

    for date in TARGET_DATES:
        start, end = date_window(date)
        collection = (ee.ImageCollection('COPERNICUS/S5P/OFFL/L3_NO2')
                      .select('tropospheric_NO2_column_number_density')
                      .filterBounds(point)
                      .filterDate(start, end))

        count = collection.size().getInfo()
        if count == 0:
            print(f'{name:15s} {date} -- no Sentinel-5P readings found in this window, try a wider WINDOW_DAYS')
            continue

        composite = collection.mean().clip(region)
        zone_images_no2[(name, date)] = composite
        print(f'{name:15s} {date} -- NO2 layer built from {count} daily readings')

print(f'\nTotal NO2 layers pulled: {len(zone_images_no2)} out of {len(zones) * len(TARGET_DATES)} possible zone-date combinations')

## Step 8 - Save everything to ONE Drive folder

**What this does:** exports every (zone, date) image pulled in Steps 5 and 7 to your Google Drive. Everything lands flat inside one folder at this stage -- **Step 8.5 right after this one sorts everything into 13 per-zone subfolders automatically**, so don't worry about the flat structure here, it's temporary.

**Why it isn't organized into subfolders at export time:** Earth Engine's export step only reliably supports a single flat folder by name -- it doesn't cleanly support creating nested subfolders during export itself. Sorting into per-zone folders as a separate step afterward is the more reliable approach, and it also means Step 8.5 can re-sort anything at any point, not just right after a fresh export.

**File naming:** `{ZoneName}_{Date}_S2.tif` and `{ZoneName}_{Date}_NO2.tif`.

**Band update (unchanged from before):** Sentinel-2 exports still pull all 6 bands Prithvi expects (`B2, B3, B4, B8A, B11, B12`).

**Do not rename `DRIVE_FOLDER` between sessions.**

Exports run in the background on Google's servers -- with 130 possible images, this can take a while to fully finish. That's expected.

In [ ]:
DRIVE_FOLDER = 'AirSentinel_Satellite_Images'  # never change this name between sessions

for (name, date), image in zone_images_s2.items():
    region = ee.Geometry.Point(zones[name]).buffer(2000).bounds()
    safe_name = name.replace(' ', '_')
    task = ee.batch.Export.image.toDrive(
        image=image.select(['B2', 'B3', 'B4', 'B8A', 'B11', 'B12']),  # Blue, Green, Red, Narrow NIR, SWIR1, SWIR2 -- Prithvi's expected 6 bands, in order
        description=f'{safe_name}_{date}_S2',
        folder=DRIVE_FOLDER,
        fileNamePrefix=f'{safe_name}_{date}_S2',
        region=region,
        scale=10,
        maxPixels=1e9
    )
    task.start()
    print(f'Started photo export: {name} {date}')

for (name, date), image in zone_images_no2.items():
    region = ee.Geometry.Point(zones[name]).buffer(5000).bounds()
    safe_name = name.replace(' ', '_')
    task = ee.batch.Export.image.toDrive(
        image=image,
        description=f'{safe_name}_{date}_NO2',
        folder=DRIVE_FOLDER,
        fileNamePrefix=f'{safe_name}_{date}_NO2',
        region=region,
        scale=1000,
        maxPixels=1e9
    )
    task.start()
    print(f'Started NO2 export: {name} {date}')

print(f'\nAll exports started ({len(zone_images_s2)} photo + {len(zone_images_no2)} NO2 tasks). Check your Google Drive folder in a few minutes: {DRIVE_FOLDER}')

## Step 8.5 - Sort everything into ONE folder, with a per-zone subfolder for each

**What this does, in two parts:**
1. **Sweep:** searches your entire Drive for any zone image, wherever it ended up, same as before.
2. **Sort:** moves every file into a subfolder named after its zone -- so inside `AirSentinel_Satellite_Images` you'll end up with 13 subfolders (`Anand_Vihar/`, `Mundka/`, ... `Dwarka/`), each containing that zone's S2 and NO2 files for every date you pulled.

Safe to run anytime, as many times as you want -- it never overwrites a file, only moves things into the correct place. The zone is figured out directly from the filename (which already starts with the zone name), so this works automatically without you telling it which file belongs where.

In [ ]:
import os
import shutil
import glob

try:
    from google.colab import drive
    drive.mount('/content/drive')
    drive_root = '/content/drive/MyDrive'
except ImportError:
    # Not running in Colab (e.g. VS Code) -- use your local Google Drive for Desktop folder instead.
    drive_root = 'PASTE_YOUR_LOCAL_GOOGLE_DRIVE_FOLDER_PATH_HERE'
    print('Not running in Colab -- using local drive_root instead. Update the path above if needed.')

target_dir = f'{drive_root}/{DRIVE_FOLDER}'
os.makedirs(target_dir, exist_ok=True)

zone_names = [z.replace(' ', '_') for z in zones.keys()]

# Make sure every zone has its own subfolder, even before any files exist for it yet
for zn in zone_names:
    os.makedirs(os.path.join(target_dir, zn), exist_ok=True)

# Find every zone file anywhere in Drive (flat in target_dir, in a wrong folder, or already correctly sorted)
all_tifs = glob.glob(f'{drive_root}/**/*.tif', recursive=True)

moved, skipped, already_sorted, failed = 0, 0, 0, []

for f in all_tifs:
    try:
        fname = os.path.basename(f)
        matching_zone = next((zn for zn in zone_names if fname.startswith(zn + '_')), None)
        if matching_zone is None:
            continue  # not one of our zone images, leave it alone

        correct_dir = os.path.join(target_dir, matching_zone)
        dest = os.path.join(correct_dir, fname)

        if os.path.abspath(f) == os.path.abspath(dest):
            already_sorted += 1
            continue

        if os.path.exists(dest):
            print(f'Skipped (duplicate already sorted): {fname}')
            skipped += 1
            continue

        shutil.move(f, dest)
        print(f'Sorted into {matching_zone}/: {fname}')
        moved += 1

    except Exception as e:
        # A single bad file (permissions, sync lag, odd name) no longer stops the rest of the sort
        failed.append((f, str(e)))
        print(f'FAILED to sort {f}: {e}')

print(f'\nDone. Moved {moved}, skipped {skipped} duplicates, {already_sorted} already correctly sorted, {len(failed)} failed.')

if failed:
    print('\nThese specific files failed -- look at the error next to each one:')
    for f, err in failed:
        print(f'  {f}\n    -> {err}')

print(f'\nEverything now lives under: {target_dir}/<zone_name>/')

## Step 9 - Look at everything properly

**What this does:** your saved files are raw scientific data, not normal photo-app-friendly images, so a regular photo viewer will show them wrong. This cell displays both layers correctly, right here -- and now searches inside all 13 per-zone subfolders automatically, not just one flat folder.

**With up to 130 images, showing everything at once would be a huge, unreadable wall of tiny pictures.** So this shows a capped sample by default (`MAX_PREVIEW`, currently 20) across all zones, plus a separate `preview_zone('Anand Vihar')` function right after it so you can pull up every date for one specific zone at a proper size whenever you want to actually check something.

In [ ]:
import numpy as np
import rasterio
import matplotlib.pyplot as plt

folder_path = f'{drive_root}/{DRIVE_FOLDER}'

s2_files = sorted(glob.glob(f'{folder_path}/**/*_S2.tif', recursive=True))
no2_files = sorted(glob.glob(f'{folder_path}/**/*_NO2.tif', recursive=True))

print(f'Found {len(s2_files)} photo(s) and {len(no2_files)} NO2 layer(s) total, across all zone folders.')

def show_grid(files, title, is_rgb):
    if not files:
        print(f'Nothing to show for: {title}')
        return
    cols = 4
    rows = max(1, (len(files) + cols - 1) // cols)
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
    axes = np.array(axes).reshape(-1)
    fig.suptitle(title, fontsize=14)

    for ax, filepath in zip(axes, files):
        with rasterio.open(filepath) as src:
            arr = src.read().astype(np.float32)

        if is_rgb:
            out = np.zeros((arr.shape[1], arr.shape[2], 3), dtype=np.uint8)
            rgb_band_indices = [2, 1, 0]  # Red, Green, Blue out of the 6-band file
            for channel, band_idx in enumerate(rgb_band_indices):
                band = arr[band_idx]
                lo, hi = np.percentile(band, 2), np.percentile(band, 98)
                band = np.clip((band - lo) / (hi - lo + 1e-6) * 255, 0, 255)
                out[:, :, channel] = band.astype(np.uint8)
            ax.imshow(out)
        else:
            band = arr[0]
            lo, hi = np.percentile(band, 2), np.percentile(band, 98)
            ax.imshow(band, cmap='YlOrRd', vmin=lo, vmax=hi)

        # Show "Zone Date" instead of the full filename, easier to read at a glance
        label = filepath.split('/')[-1].replace('.tif', '').rsplit('_', 1)[0]
        ax.set_title(label, fontsize=9)
        ax.axis('off')

    for ax in axes[len(files):]:
        ax.axis('off')
    plt.tight_layout()
    plt.show()

MAX_PREVIEW = 20  # raise this if you want a bigger sample -- just gets slow/unwieldy past ~30-40

show_grid(s2_files[:MAX_PREVIEW], f'Sentinel-2 -- sample of {min(MAX_PREVIEW, len(s2_files))} of {len(s2_files)} total', is_rgb=True)
show_grid(no2_files[:MAX_PREVIEW], f'Sentinel-5P -- sample of {min(MAX_PREVIEW, len(no2_files))} of {len(no2_files)} total', is_rgb=False)

def preview_zone(zone_name):
    """Show every date pulled for one specific zone, at a readable size."""
    safe_name = zone_name.replace(' ', '_')
    zone_s2 = sorted(glob.glob(f'{folder_path}/{safe_name}/*_S2.tif'))
    zone_no2 = sorted(glob.glob(f'{folder_path}/{safe_name}/*_NO2.tif'))
    print(f'{zone_name}: {len(zone_s2)} photo(s), {len(zone_no2)} NO2 layer(s)')
    show_grid(zone_s2, f'{zone_name} -- all dates (photo)', is_rgb=True)
    show_grid(zone_no2, f'{zone_name} -- all dates (NO2)', is_rgb=False)

# Example -- change the zone name to check any of your 13
preview_zone('Anand Vihar')

## Step 10 - Prepare the labels table (placeholder, fill once your teammate's data is ready)

**What this does, and why:** Prithvi's fine-tuning step needs an actual label per image -- and that label should come from real monitoring-station data, not a visual guess. This cell builds the empty structure now, with **one row per (zone, date) combination actually pulled** -- so if you widened `TARGET_DATES` in Step 4.5, this table grows to match, giving you many more rows to fine-tune on than the old one-row-per-zone version.

`load_caaqms_labels()` below is a stand-in. Replace its contents with actual code that reads whatever file/table your teammate gives you (likely a CSV with columns for zone, date, and dominant pollutant) -- it needs to match on both zone AND date now, not just zone.

In [ ]:
import pandas as pd

# Build one row per (zone, date) combination that was actually pulled in Step 5
# Note: s2_file / no2_file now include the zone subfolder in the path, e.g. 'Anand_Vihar/Anand_Vihar_2026-06-01_S2.tif'
# -- since Step 8.5 now sorts every file into a per-zone subfolder, not a flat folder.
rows = []
for (name, date) in zone_images_s2.keys():
    safe_name = name.replace(' ', '_')
    rows.append({
        'zone': name,
        'date': date,
        's2_file': f'{safe_name}/{safe_name}_{date}_S2.tif',
        'no2_file': f'{safe_name}/{safe_name}_{date}_NO2.tif' if (name, date) in zone_images_no2 else None,
        'dominant_pollutant': None,  # <- to be filled in from CAAQMS data
        'label_source': None,        # <- e.g. 'CAAQMS' or 'Supersite', once known
    })

labels_table = pd.DataFrame(rows)
print(f'{len(labels_table)} zone-date rows ready for labeling.')

def load_caaqms_labels():
    """
    STAND-IN FUNCTION -- replace this once your teammate's CAAQMS data is ready.
    Should return a DataFrame with at least: zone, date, dominant_pollutant.
    Must match on BOTH zone and date now that there are multiple dates per zone.
    Example once ready:
        return pd.read_csv(f'{drive_root}/AirSentinel_Satellite_Images/caaqms_labels.csv')
    """
    return pd.DataFrame(columns=['zone', 'date', 'dominant_pollutant'])

caaqms_labels = load_caaqms_labels()
if not caaqms_labels.empty:
    labels_table = labels_table.drop(columns=['dominant_pollutant']).merge(
        caaqms_labels, on=['zone', 'date'], how='left'
    )
    print('Merged real CAAQMS labels in.')
else:
    print('No CAAQMS labels available yet -- table below has empty label columns, ready to merge later.')

labels_table

## Step 10.5 — PLACEHOLDER labels, for testing only

**What this does, and why:** your real labels aren't ready yet, but there's no reason to sit idle — this fills the label column with **random**, fake categories, just so you can run the fine-tuning pipeline end to end and confirm the code itself works (loads images, trains, saves a model file) before real data exists.

**This is not a real model once you do this — don't mistake it for one.** A model trained on random labels learns nothing meaningful; it only proves your code doesn't crash. The random seed below makes it reproducible, and every placeholder row is clearly flagged so it can't quietly get treated as real data later.

**Once your teammate's CAAQMS data is ready:** don't touch this cell — just finish `load_caaqms_labels()` in Step 10 with their real data and re-run Step 10. That will overwrite these placeholders with real labels automatically.

In [ ]:
import random

USING_PLACEHOLDER_LABELS = labels_table['dominant_pollutant'].isna().all()

if USING_PLACEHOLDER_LABELS:
    print('WARNING: using random PLACEHOLDER labels — for pipeline testing only, not a real model.\n')

    categories = ['dust', 'crop_burning_smoke', 'industrial_haze', 'traffic_heavy', 'clear']
    random.seed(42)  # fixed seed = same random labels every time you run this, so results are reproducible

    labels_table['dominant_pollutant'] = [random.choice(categories) for _ in range(len(labels_table))]
    labels_table['label_source'] = 'PLACEHOLDER — random, not real'
else:
    print('Real labels already present — skipping placeholder generation.')

labels_table

## Step 11 - Multiply your real training data through augmentation

**What this does, and why it's legitimate, not cheating:** satellite images have no "correct" orientation -- a photo of a zone flipped left-right or rotated 90 degrees is still a completely real, valid image of that same zone. This cell takes every image that has a real (non-placeholder) label and generates 7 more versions of it -- 3 rotations, a horizontal flip, and that flip rotated 3 more ways -- covering every unique orientation. Each version gets the exact same label as the original, since flipping/rotating a photo of dust doesn't turn it into a photo of traffic.

**This is standard practice for small image datasets, not a workaround.** It won't fix a fundamentally wrong label, but it does give the model many more real pixel-pattern variations to learn from instead of memorizing 226 exact images.

**Safety:** this only touches images with real `CAAQMS_heuristic` (or `Supersite`) labels -- placeholder-labeled images are never multiplied, since that would just multiply meaningless labels. Output goes to a **new file**, `labels_augmented.csv` -- your original `labels.csv` is never modified, so you can always go back to the un-augmented version if needed. Safe to re-run -- it skips any augmented file that already exists rather than duplicating it.

**One honest technical caveat:** the augmented `.tif` files keep a copy of the original's geospatial metadata for convenience, but that metadata is no longer strictly accurate for a rotated/flipped image (the real-world location a rotated pixel "points to" isn't where the file claims). This doesn't matter for training -- Prithvi consumes the pixel values, not the geographic metadata -- but these augmented files should never be used for any actual geospatial/mapping purpose, only for training.

In [ ]:
import numpy as np
import pandas as pd
import rasterio
import os

labels_path = f'{target_dir}/labels.csv'
df = pd.read_csv(labels_path)

# Only augment rows with a real label -- never multiply placeholder/random labels
real_mask = df['label_source'].astype(str).str.startswith(('CAAQMS_heuristic', 'Supersite'))
real_rows = df[real_mask].copy()
print(f'{len(df)} total rows in labels.csv -- {len(real_rows)} have real labels and will be augmented.')

# The 7 non-identity transforms of a square image's symmetry group (dihedral group of order 8).
# Each is a (rot90_k, flip_first) pair: rotate by 90*k degrees, optionally flip horizontally first.
TRANSFORMS = {
    'rot90':        (1, False),
    'rot180':       (2, False),
    'rot270':       (3, False),
    'flipH':        (0, True),
    'flipH_rot90':  (1, True),
    'flipH_rot180': (2, True),
    'flipH_rot270': (3, True),
}

def augment_array(arr, rot_k, flip_first):
    # arr shape: (bands, H, W)
    out = arr
    if flip_first:
        out = out[:, :, ::-1]  # flip along width (horizontal flip)
    if rot_k:
        out = np.rot90(out, k=rot_k, axes=(1, 2))
    return out

def augment_file(src_path, dest_path, rot_k, flip_first):
    if os.path.exists(dest_path):
        return False  # already done, safe to re-run this cell
    with rasterio.open(src_path) as src:
        arr = src.read()
        profile = src.profile.copy()

    out_arr = augment_array(arr, rot_k, flip_first)

    # Width/height swap for 90/270 degree rotations
    if rot_k in (1, 3):
        profile['width'], profile['height'] = profile['height'], profile['width']

    os.makedirs(os.path.dirname(dest_path), exist_ok=True)
    with rasterio.open(dest_path, 'w', **profile) as dst:
        dst.write(out_arr)
    return True

new_rows = []
created, skipped_existing, failed = 0, 0, []

for _, row in real_rows.iterrows():
    for suffix, (rot_k, flip_first) in TRANSFORMS.items():
        new_row = row.copy()

        for col in ['s2_file', 'no2_file']:
            orig = row.get(col)
            if pd.isna(orig) or not orig:
                continue
            src_path = f'{target_dir}/{orig}'
            dest_rel = orig.replace('.tif', f'_aug-{suffix}.tif')
            dest_path = f'{target_dir}/{dest_rel}'

            try:
                made = augment_file(src_path, dest_path, rot_k, flip_first)
                new_row[col] = dest_rel
                if made:
                    created += 1
                else:
                    skipped_existing += 1
            except Exception as e:
                failed.append((src_path, str(e)))
                new_row[col] = None

        new_row['label_source'] = f"{row['label_source']} [augmented: {suffix}]"
        new_rows.append(new_row)

augmented_df = pd.DataFrame(new_rows)
combined_df = pd.concat([df, augmented_df], ignore_index=True)

out_path = f'{target_dir}/labels_augmented.csv'
combined_df.to_csv(out_path, index=False)

print(f'\nFiles created: {created}, already existed (skipped): {skipped_existing}, failed: {len(failed)}')
if failed:
    print('\nFailed files -- check these:')
    for path, err in failed:
        print(f'  {path} -> {err}')

print(f'\nOriginal labels.csv: {len(df)} rows (unchanged, not touched)')
print(f'New labels_augmented.csv: {len(combined_df)} rows ({len(df)} original + {len(augmented_df)} augmented)')
print(f'\nPoint train_prithvi.py at labels_augmented.csv instead of labels.csv to use the larger dataset.')

## What's next

This notebook stops here on purpose — everything past this point is Phase 2 (hand-checking labels once real CAAQMS data is merged in, then fine-tuning Prithvi), which deserves its own notebook rather than being bolted onto this one.

Before moving on, confirm:
1. Step 9 shows a clean photo *and* a clean NO2 heatmap for as many zone-date combinations as possible — widen `WINDOW_DAYS` (Step 4.5) and re-run Steps 5/7/8 for any that are missing.
2. You've synced with your teammate on the exact format their CAAQMS data will arrive in, so `load_caaqms_labels()` above is a five-minute edit, not a rebuild.
3. If you want more training samples, just add more dates to `TARGET_DATES` in Step 4.5 and re-run Steps 5, 7, 8, 8.5, 9, and 10 — everything downstream already scales automatically with however many dates you give it.